<a href="https://colab.research.google.com/github/aapaul29/Distance-Recognition-SML1-/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SML Project 1
This notebook works in both **VS Code (local)** and **Google Colab**.

## 1. Environment Detection & Setup

In [46]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules
print(f"Running in {'Google Colab' if IN_COLAB else 'VS Code / local'}")

Running in VS Code / local


## 2. Google Colab: Mount Drive & Navigate to Project
This cell only runs on Colab. Adjust `COLAB_PROJECT_PATH` to where you uploaded the project folder in your Google Drive.

In [47]:
# ---------------------------------------------------------------
# COLAB ONLY — skip if running locally
# Adjust this path to match your Google Drive folder structure.
COLAB_PROJECT_PATH = "/content/drive/MyDrive/Aaryan/Uni/BS Mech Eng ETH Zurich/2nd year/Semester 4/Stochastics & Machine Learning/Distance-Recognition-SML1-"
# ---------------------------------------------------------------

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd $COLAB_PROJECT_PATH
    print(f"Working directory: {os.getcwd()}")

    # Unzip data if not already extracted
    data_zip = os.path.join(COLAB_PROJECT_PATH, "data.zip")
    data_dir = os.path.join(COLAB_PROJECT_PATH, "data")
    if os.path.exists(data_zip) and not os.path.exists(data_dir):
        print("Unzipping data...")
        os.makedirs(data_dir, exist_ok=True)
        !unzip -q $data_zip -d $data_dir
        print("Done.")
    else:
        print("Data directory already exists or no zip found — skipping unzip.")
else:
    print("Local environment — no mounting needed.")

Local environment — no mounting needed.


## 3. Install / Verify Dependencies

In [48]:
# On Colab, install any missing packages from requirements.txt.
# Locally, this is a no-op if your conda env is already set up.
if IN_COLAB:
    !pip install -q -r requirements.txt

## 4. Auto-reload & Imports

In [49]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from utils import load_config, load_dataset, load_test_dataset, print_results, save_results

# Add further sklearn imports here as needed.
# Note: SVRs are NOT allowed in this project.

## 5. Load Config & Data

In [50]:
config = load_config()

images, distances = load_dataset(config, split="train")
print(f"[INFO]: Dataset loaded with {len(images)} samples.")
print(f"[INFO]: Feature dimensionality: {images.shape[1]}")

[INFO]: Configs are loaded with: 
 {'data_dir': WindowsPath('data'), 'load_rgb': True, 'downsample_factor': 2}
[INFO]: Dataset loaded with 3000 samples.
[INFO]: Feature dimensionality: 67500


## 6. Your Implementation
Implement preprocessing, model training, and evaluation below.

In [51]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np, time

ds = config["downsample_factor"]
H = W = 300 // ds
C = 3 if config["load_rgb"] else 1
print(f"Image size: {H}x{W}, channels: {C}")

def to_gray(img_flat):
    if C == 3:
        return img_flat.reshape(H, W, 3).mean(axis=2).astype(np.float32)
    return img_flat.reshape(H, W).astype(np.float32)

def sobel_axis1(img):
    p = np.pad(img, 1, mode='reflect')
    return (
        -p[0:-2, 0:-2] - 2*p[1:-1, 0:-2] - p[2:, 0:-2]
        + p[0:-2,  2:]  + 2*p[1:-1,  2:]  + p[2:,  2:]
    ).astype(np.float32)

def sobel_axis0(img):
    p = np.pad(img, 1, mode='reflect')
    return (
        -p[0:-2, 0:-2] - 2*p[0:-2, 1:-1] - p[0:-2, 2:]
        + p[2:,  0:-2]  + 2*p[2:,  1:-1]  + p[2:,  2:]
    ).astype(np.float32)

def extract_lbp(img_2d):
    lbp = np.zeros_like(img_2d, dtype=np.uint8)
    center = img_2d[1:-1, 1:-1]
    for i, n in enumerate([
        img_2d[0:-2,0:-2], img_2d[0:-2,1:-1], img_2d[0:-2,2:],
        img_2d[1:-1,2:], img_2d[2:,2:], img_2d[2:,1:-1],
        img_2d[2:,0:-2], img_2d[1:-1,0:-2]
    ]):
        lbp[1:-1,1:-1] += (n >= center).astype(np.uint8) * (2**i)
    hist, _ = np.histogram(lbp, bins=32, range=(0,256))
    return hist.astype(np.float32) / (hist.sum() + 1e-8)

def extract_all_features(X_raw, H, W, C):
    """The 9.7cm feature set."""
    all_feat = []
    for img in X_raw:
        gray = to_gray(img)
        wt = np.linspace(0.3, 1.0, H).reshape(-1, 1)
        img_w = gray * wt
        feats = []

        # 1. Grid 4x4 + 5x5
        for g in [4, 5]:
            gh, gw = H // g, W // g
            for i in range(g):
                for j in range(g):
                    c = img_w[i*gh:(i+1)*gh, j*gw:(j+1)*gw]
                    feats.extend([c.mean(), c.var()])

        # 2. Strips (25)
        n_strips = 25
        sh = H // n_strips
        means = []
        for i in range(n_strips):
            s = img_w[i*sh:(i+1)*sh, :]
            m = s.mean(); means.append(m)
            feats.extend([m, s.var(), s.min(), s.max()])
        for i in range(1, n_strips):
            feats.append(means[i] - means[i-1])

        # 3. Vertical profile
        row_means = gray.mean(axis=1)
        vg = np.diff(row_means)
        feats.extend([vg.mean(), vg.std(), vg.min(), vg.max()])
        feats.append(np.argmax(np.abs(vg)) / H)
        feats.append(np.argmin(vg) / H)
        feats.append(np.argmax(row_means) / H)
        feats.append(np.argmin(row_means) / H)
        for p in [25, 50, 75]:
            feats.append(np.percentile(row_means, p))
        feats.append(row_means[:H//3].mean() - row_means[2*H//3:].mean())

        # 4. Bottom-third grid
        bt = 2*H//3
        bot = img_w[bt:, :]
        bh = (H - bt) // 4; bw = W // 4
        if bh > 0:
            for i in range(4):
                for j in range(4):
                    c = bot[i*bh:(i+1)*bh, j*bw:(j+1)*bw]
                    feats.extend([c.mean(), c.var()])

        # 5. Histogram
        hg, _ = np.histogram(gray, bins=16, range=(0,255))
        feats.extend((hg/(hg.sum()+1e-8)).tolist())

        # 6. Global
        feats.extend([gray.mean(), gray.std(), gray.var()])
        for p in [5,10,25,50,75,90,95]:
            feats.append(np.percentile(gray, p))
        feats.append(gray[H//2:].mean() / (gray[:H//2].mean()+1e-8))
        feats.append(gray[:H//3].mean() / (gray[2*H//3:].mean()+1e-8))
        feats.append(gray[2*H//3:].std() / (gray[:H//3].std()+1e-8))

        # 7. LBP
        feats.extend(extract_lbp(gray).tolist())

        # 8. Sobel
        sx = sobel_axis1(img_w); sy = sobel_axis0(img_w)
        feats.extend([sx.mean(), sx.var(), sy.mean(), sy.var()])
        hsx, _ = np.histogram(sx, bins=16)
        hsy, _ = np.histogram(sy, bins=16)
        feats.extend((hsx/(hsx.sum()+1e-8)).tolist())
        feats.extend((hsy/(hsy.sum()+1e-8)).tolist())

        # 9. Floor scan (8 thresholds)
        grad = np.abs(np.diff(gray, axis=0))
        for t in [3,5,8,10,15,20,25,35]:
            rows = np.where(grad > t, np.arange(H-1).reshape(-1,1), -1).max(axis=0)
            v = np.where(rows >= 0, rows / H, 0).astype(np.float32)
            valid = v[v > 0]
            if len(valid) > 0:
                feats.extend([np.median(valid), np.mean(valid), np.std(valid),
                              np.percentile(valid,25), np.percentile(valid,75)])
            else:
                feats.extend([0,0,0,0,0])

        # 10. Edge zones (10)
        mag = np.sqrt(sx**2 + sy**2)
        zh = H // 10
        for i in range(10):
            zone = mag[i*zh:(i+1)*zh]
            feats.extend([zone.mean(), zone.var()])

        # 11. Horizontal
        col_means = gray.mean(axis=0)
        cg = np.diff(col_means)
        feats.extend([col_means.mean(), col_means.std(), cg.mean(), cg.std()])
        feats.append(gray[:,:W//2].mean() / (gray[:,W//2:].mean()+1e-8))
        for p in [10,25,50,75,90]:
            feats.append(np.percentile(np.min(gray, axis=0), p))
        for p in [10,25,50,75,90]:
            feats.append(np.percentile(gray[H//2:].mean(axis=0), p))

        # 12. RGB
        if C == 3:
            img_rgb = img.reshape(H,W,3).astype(np.float32)
            r,g,b = img_rgb[:,:,0], img_rgb[:,:,1], img_rgb[:,:,2]
            for ch in [r,g,b]:
                feats.extend([ch.mean(), ch.std()])
            feats.extend([r.mean()/(g.mean()+1e-8), r.mean()/(b.mean()+1e-8), g.mean()/(b.mean()+1e-8)])
            for ch in [r,g,b]:
                feats.append(ch[H//2:].mean()/(ch[:H//2].mean()+1e-8))
            for ch in [r,g,b]:
                hc, _ = np.histogram(ch, bins=16, range=(0,255))
                feats.extend((hc/(hc.sum()+1e-8)).tolist())
            sh10 = H // 10
            for ch in [r,g,b]:
                for i in range(10):
                    feats.append(ch[i*sh10:(i+1)*sh10].mean())
            for ch in [r,g,b]:
                feats.append(ch[2*H//3:].mean() / (ch[:H//3].mean()+1e-8))

        all_feat.append(np.array(feats, dtype=np.float32))
    return np.array(all_feat)

# Split
X_train_raw, X_val_raw, y_train, y_val = train_test_split(
    images, distances, test_size=0.15, random_state=42
)

t0 = time.time()
X_train_hand = extract_all_features(X_train_raw, H, W, C)
X_val_hand = extract_all_features(X_val_raw, H, W, C)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_hand)
X_val_s = scaler.transform(X_val_hand)
print(f"Done ({time.time()-t0:.1f}s), features: {X_train_s.shape[1]}")

Image size: 150x150, channels: 3
Done (148.3s), features: 515


In [52]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error
from scipy.optimize import minimize
import numpy as np

# PCA before KNN — reduces 515D curse of dimensionality
pca = PCA(n_components=70)
X_train_pca = pca.fit_transform(X_train_s)
X_val_pca   = pca.transform(X_val_s)

# KNN on PCA space (no k=1)
knn3 = KNeighborsRegressor(n_neighbors=3, weights='distance')
knn3.fit(X_train_pca, y_train)
knn5 = KNeighborsRegressor(n_neighbors=5, weights='distance')
knn5.fit(X_train_pca, y_train)

# ExtraTrees on full feature space
print("Training ExtraTrees...")
et = ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1)
et.fit(X_train_s, y_train)

# HGBRegressor — direct regression, no binning trick
print("Training HGBRegressor...")
hgb = HistGradientBoostingRegressor(max_iter=500, max_depth=5, learning_rate=0.05,
                                     min_samples_leaf=15, l2_regularization=0.2, random_state=42)
hgb.fit(X_train_s, y_train)

# Validation predictions
p_knn3 = knn3.predict(X_val_pca)
p_knn5 = knn5.predict(X_val_pca)
p_et   = et.predict(X_val_s)
p_hgb  = hgb.predict(X_val_s)

print(f"KNN k=3 (PCA-70): {mean_absolute_error(y_val, p_knn3)*100:.2f} cm")
print(f"KNN k=5 (PCA-70): {mean_absolute_error(y_val, p_knn5)*100:.2f} cm")
print(f"ExtraTrees:       {mean_absolute_error(y_val, p_et)*100:.2f} cm")
print(f"HGBRegressor:     {mean_absolute_error(y_val, p_hgb)*100:.2f} cm")

# Optimise blend weights via Nelder-Mead on val set
preds_val = np.stack([p_knn3, p_knn5, p_et, p_hgb], axis=1)

def blend_mae(w):
    w = np.abs(w) / (np.abs(w).sum() + 1e-8)
    return mean_absolute_error(y_val, preds_val @ w)

res = minimize(blend_mae, [0.25]*4, method='Nelder-Mead',
               options={'maxiter': 10000, 'xatol': 1e-6, 'fatol': 1e-6})
w_opt = np.abs(res.x) / np.abs(res.x).sum()
print(f"\nOptimal weights — KNN3: {w_opt[0]:.3f}, KNN5: {w_opt[1]:.3f}, ET: {w_opt[2]:.3f}, HGB: {w_opt[3]:.3f}")

val_pred = preds_val @ w_opt
print(f"Blend MAE: {mean_absolute_error(y_val, val_pred)*100:.2f} cm")

Training ExtraTrees...
Training HGBRegressor...
KNN k=3 (PCA-70): 10.78 cm
KNN k=5 (PCA-70): 11.22 cm
ExtraTrees:       12.21 cm
HGBRegressor:     11.99 cm

Optimal weights — KNN3: 0.637, KNN5: 0.000, ET: 0.000, HGB: 0.363
Blend MAE: 10.11 cm


In [53]:
print("=== Validation Set ===")
print_results(y_val, val_pred)

=== Validation Set ===
MAE: 10.11
R2: 88.399


## 7. Predict on Test Set & Save

In [54]:
print("Retraining on all 3000 samples...")
X_all_hand = extract_all_features(images, H, W, C)
sc_f = StandardScaler()
X_all_s = sc_f.fit_transform(X_all_hand)

pca_f = PCA(n_components=70)
X_all_pca = pca_f.fit_transform(X_all_s)

knn3_f = KNeighborsRegressor(n_neighbors=3, weights='distance')
knn3_f.fit(X_all_pca, distances)
knn5_f = KNeighborsRegressor(n_neighbors=5, weights='distance')
knn5_f.fit(X_all_pca, distances)

print("Training ExtraTrees...")
et_f = ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1)
et_f.fit(X_all_s, distances)

print("Training HGBRegressor...")
hgb_f = HistGradientBoostingRegressor(max_iter=500, max_depth=5, learning_rate=0.05,
                                       min_samples_leaf=15, l2_regularization=0.2, random_state=42)
hgb_f.fit(X_all_s, distances)

test_raw = np.array(load_test_dataset(config))
X_test_hand = extract_all_features(test_raw, H, W, C)
X_test_s    = sc_f.transform(X_test_hand)
X_test_pca  = pca_f.transform(X_test_s)

t_knn3 = knn3_f.predict(X_test_pca)
t_knn5 = knn5_f.predict(X_test_pca)
t_et   = et_f.predict(X_test_s)
t_hgb  = hgb_f.predict(X_test_s)

preds_test = np.stack([t_knn3, t_knn5, t_et, t_hgb], axis=1)
test_pred  = preds_test @ w_opt
test_pred  = np.clip(test_pred, 0.1, 4.0)
save_results(test_pred)
print(f"Saved! Range: {test_pred.min():.3f} - {test_pred.max():.3f} m")

Retraining on all 3000 samples...
Training ExtraTrees...
Training HGBRegressor...
Saved! Range: 0.419 - 2.674 m
